# Model comparison, frozen evaluation and export
    ## Objective
    Compare completed experiments using validation evidence, freeze the winning choice, then evaluate held-out test exactly once per frozen checkpoint.
    Cached test receipts prevent routine reruns. Never tune after viewing test results.

In [ ]:
import sys
import torch
import torchvision
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU-ONLY TRAINING: CUDA unavailable in this kernel. Fix NVIDIA driver, CUDA wheel, interpreter or notebook kernel. CPU training is forbidden.")
device = torch.device("cuda:0")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU properties:", torch.cuda.get_device_properties(0))
x = torch.randn((128, 128), device=device)
y = x @ x
assert y.is_cuda and y.device.type == "cuda"
torch.cuda.synchronize()
print("GPU tensor test PASSED:", y.device)
del x, y

## Setup and load authentic run histories

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open the FaceAttend project folder.")
sys.path.insert(0, str(ROOT / "src"))
os.environ["TORCH_HOME"] = str(ROOT / ".cache/torch")
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt
from faceattend.utils import environment, load_config, seed_everything, write_json

In [ ]:
from faceattend.evaluation import EXPERIMENTS, freeze_selection, evaluate_and_export
from faceattend.metrics import plot_history, save_metrics
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for experiment in EXPERIMENTS:
    folder = ROOT / "outputs" / experiment
    if not (folder / "completed.json").exists():
        raise RuntimeError(f"TRAINING NOT COMPLETED: {experiment}. No comparison metrics may be invented.")
    history = pd.read_csv(folder / "history.csv")
    color, style = ("#1f77b4", "-") if experiment == "m0_pretrained_baseline" else ("#ff7f0e", "--")
    axes[0].plot(history.epoch, history.val_macro_f1, label=experiment, color=color, linestyle=style)
    axes[1].plot(history.epoch, history.val_acc, label=experiment, color=color, linestyle=style)
for axis, label in zip(axes, ["Validation macro F1", "Validation accuracy"]):
    axis.set(xlabel="Epoch", ylabel=label, title=label)
    axis.legend()
fig.suptitle("Source: authentic per-run history.csv files")
fig.tight_layout()
(ROOT / "outputs/comparison").mkdir(parents=True, exist_ok=True)
fig.savefig(ROOT / "outputs/comparison/validation_curves.png", dpi=140)
plt.show()

## Freeze the validation decision
Assert matching controlled settings and manifests. Winner uses macro F1, then accuracy, then lower loss.

In [ ]:
frozen = freeze_selection(ROOT)
display(pd.DataFrame(frozen["models"]))
print("Frozen winner:", frozen["winner"])
for row in frozen["models"]:
    checkpoint = torch.load(ROOT / "outputs" / row["model"] / "best.pt", map_location="cpu", weights_only=True)
    save_metrics(checkpoint["validation_metrics"], ROOT / "outputs/comparison", row["model"] + "_validation")
    plt.show()

## Held-out evaluation, warmed GPU latency and portable export
Test results cannot change the already saved winner. Batch-1 FP32 latency uses 20 warm-ups, 100 synchronized forwards and reports median and p95; excludes image preprocessing and host transfer.

In [ ]:
comparison = evaluate_and_export(ROOT, device)
display(comparison)
print("Export:", ROOT / "models/faceattend_best.pt")

## Interpretation
    Discuss validation and test accuracy, macro F1, weak classes, latency and peak training memory from the authentic comparison table.
    Do not promise that M1 wins. The portable checkpoint carries architecture, class order and exact preprocessing;
    inference reconstructs the network without downloading initial weights. Results cannot establish true emotional state.